In [1]:
from utils import Helper
from pathlib import Path
import os, re, time
utils = Helper()

from docAI import PageWordExtractor
from google.cloud import documentai
from google.protobuf.json_format import MessageToDict
import pandas as pd
import json
from konstant import SECRET_KEY_GOOGLE, DOCUMENT_AI_PROJECT
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SECRET_KEY_GOOGLE

In [3]:


PROJECT_ID = "502266563041"
LOCATION = "us"
PROCESSOR_ID = "9df037bb81be425d"

client = documentai.DocumentProcessorServiceClient()

name = client.processor_path(
    PROJECT_ID,
    LOCATION,
    PROCESSOR_ID,
)

with open(r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\eda_pdf\pdf_batch\batch_2.pdf", "rb") as f:
    pdf_bytes = f.read()

request = documentai.ProcessRequest(
    name=name,
    raw_document={
        "content": pdf_bytes,
        "mime_type": "application/pdf"
    }
)

result = client.process_document(request=request)
doc = result.document



doc_json = MessageToDict(doc._pb)

with open("document_ai_output.json", "w", encoding="utf-8") as f:
    json.dump(doc_json, f, indent=2)

print("Saved document_ai_output.json")

def get_text(layout, full_text):
    response = ""

    if not layout.text_anchor.text_segments:
        return ""

    for segment in layout.text_anchor.text_segments:
        start = int(segment.start_index) if segment.start_index else 0
        end = int(segment.end_index)

        response += full_text[start:end]

    return response.strip()



all_tables = []
for page_num, page in enumerate(doc.pages, start=1):
    print(
        f"Page {page_num} | Tables Found = {len(page.tables)}"
    )

    for table_num, table in enumerate(page.tables, start=1):

        rows = []

        # header rows
        for row in table.header_rows:

            rows.append([
                get_text(cell.layout, doc.text)
                for cell in row.cells
            ])

        # body rows
        for row in table.body_rows:

            rows.append([
                get_text(cell.layout, doc.text)
                for cell in row.cells
            ])

        df = pd.DataFrame(rows)

        all_tables.append(df)

        print(
            f"\nPage {page_num} Table {table_num}"
        )
        print(df.head())

        # save html
        html_file = f"page_{page_num}_table_{table_num}.html"

        df.to_html(
            html_file,
            index=False,
            border=1
        )

        print(f"Saved {html_file}")


if all_tables:

    with pd.ExcelWriter(
        "document_tables.xlsx",
        engine="openpyxl"
    ) as writer:

        for idx, df in enumerate(all_tables, start=1):

            df.to_excel(
                writer,
                sheet_name=f"Table_{idx}",
                index=False
            )

    print("Saved document_tables.xlsx")

Saved document_ai_output.json


In [ ]:
folder_path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\batch_google"

pdfs = os.listdir(folder_path)[100:]
for i,pdf in enumerate(pdfs):
    print(f"PDF: {i+1/ len(pdfs)}")
    file_path = os.path.join(folder_path, pdf)
    fp = Path(file_path)
    parser = PageWordExtractor(file_path)
    time.sleep(5)
    content = parser.pdf_handler(file_path)
    utils.save_json(content,f"{fp.stem}.json")

In [ ]:
import os
import pandas as pd
import fitz

excel_file = "input.xlsx"
pdf_folder = r"C:\Users\kaustubh.keny\Documents\Quarterly Results 2026 Q1"
output_folder = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\batch_google"
os.makedirs(output_folder, exist_ok=True)
df = pd.read_excel(excel_file)

batch_size = 10
batch_num = 1

current_batch = fitz.open()
pages_in_batch = 0

for _, row in df.iterrows():

    pdf_name = str(row["pdf"]).strip()
    page_no = int(row["page_n"])

    pdf_path = os.path.join(pdf_folder, f"{pdf_name}.pdf")

    if not os.path.exists(pdf_path):
        print(f"Missing PDF: {pdf_path}")
        continue

    src_doc = fitz.open(pdf_path)

    src_page_idx = page_no - 1

    if src_page_idx < 0 or src_page_idx >= len(src_doc):
        print(f"Invalid page {page_no} in {pdf_name}")
        src_doc.close()
        continue

    # Copy page into current batch
    current_batch.insert_pdf(
        src_doc,
        from_page=src_page_idx,
        to_page=src_page_idx
    )

    # Last inserted page
    page = current_batch[-1]

    header = f"{pdf_name}.pdf | Page {page_no}"

    # White background box
    page.draw_rect(
        fitz.Rect(5, 2, 250, 20),
        color=(1, 1, 1),
        fill=(1, 1, 1)
    )

    # Header text
    page.insert_text(
        (10, 14),
        header,
        fontsize=20,
        fontname="helv",
        color=(0, 0, 0)
    )

    src_doc.close()

    pages_in_batch += 1

    # Save batch
    if pages_in_batch == batch_size:

        out_file = os.path.join(
            output_folder,
            f"batch_{batch_num}.pdf"
        )

        current_batch.save(
            out_file,
            garbage=4,
            clean=True,
            deflate=True,
            deflate_images=True,
            deflate_fonts=True,
            use_objstms=1
        )

        size_mb = os.path.getsize(out_file) / (1024 * 1024)

        print(
            f"Saved {out_file} "
            f"({size_mb:.2f} MB)"
        )

        current_batch.close()

        batch_num += 1
        pages_in_batch = 0
        current_batch = fitz.open()

# Save leftover pages
if pages_in_batch > 0:

    out_file = os.path.join(
        output_folder,
        f"batch_{batch_num}.pdf"
    )

    current_batch.save(
        out_file,
        garbage=4,
        clean=True,
        deflate=True,
        deflate_images=True,
        deflate_fonts=True,
        use_objstms=1
    )

    size_mb = os.path.getsize(out_file) / (1024 * 1024)

    print(
        f"Saved {out_file} "
        f"({size_mb:.2f} MB)"
    )

    current_batch.close()

print("Done!")

In [ ]:
import os
import fitz

pdf_folder = r"C:\Users\kaustubh.keny\Documents\Quarterly Results 2026 Q1"
output_folder = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\batch_google"

os.makedirs(output_folder, exist_ok=True)

batch_size = 10  # pages per batch
batch_num = 1

current_batch = fitz.open()
pages_in_batch = 0

for pdf_name in pdf_list:

    pdf_path = os.path.join(pdf_folder, pdf_name)

    if not os.path.exists(pdf_path):
        print(f"Missing PDF: {pdf_path}")
        continue

    try:
        src_doc = fitz.open(pdf_path)

        for page_num in range(len(src_doc)):

            current_batch.insert_pdf(
                src_doc,
                from_page=page_num,
                to_page=page_num
            )

            pages_in_batch += 1

            # Save every 10 pages
            if pages_in_batch == batch_size:

                out_file = os.path.join(
                    output_folder,
                    f"batch_{batch_num}.pdf"
                )

                current_batch.save(out_file)
                current_batch.close()

                print(f"Saved {out_file}")

                batch_num += 1
                pages_in_batch = 0
                current_batch = fitz.open()

        src_doc.close()

    except Exception as e:
        print(f"Error processing {pdf_name}: {e}")

# Save remaining pages
if pages_in_batch > 0:

    out_file = os.path.join(
        output_folder,
        f"batch_{batch_num}.pdf"
    )

    current_batch.save(out_file)
    current_batch.close()

    print(f"Saved {out_file}")

print("Done!")